# Data Preparation and Baseline Evaluation

**Steps:**
1. Load and preprocess the SFT dataset (`sherry0213/MedCRAFT`)
2. Load and preprocess the DPO dataset (`sherry0213/MedCRAFT` DPO split)
3. Define 10 test prompts and save them
4. Run the base model (`TinyLlama/TinyLlama_v1.1`) on all 10 prompts
5. Compute BLEU + BERTScore against ChatGPT gold answers
6. Save all outputs for SFT and DPO

---
**Dataset:**
- **SFT:** `sherry0213/MedCRAFT` - 38k constraint-rich medical instruction-response pairs, pre-split train/eval, `instruction`/`response`/`diff` columns, professionally curated for medical LLM training
- **DPO:** `sherry0213/MedCRAFT` (medcraft_dpo_2k.jsonl) — 2k medical preference pairs from the same source
- **Base model:** `TinyLlama/TinyLlama_v1.1`

In [1]:
import subprocess
subprocess.run(["pip", "install", "datasets", "transformers", "accelerate",
                "sacrebleu", "bert-score", "torch", "pandas", "tqdm", "-q"],
               check=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

CompletedProcess(args=['pip', 'install', 'datasets', 'transformers', 'accelerate', 'sacrebleu', 'bert-score', 'torch', 'pandas', 'tqdm', '-q'], returncode=0)

In [2]:
import json
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device: {DEVICE}")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
Device: cuda



## 1. SFT Dataset - sherry0213/MedCRAFT

In [3]:
# Load SFT dataset from MedCRAFT
print("Loading SFT train split...")
sft_train_raw = load_dataset(
    "json",
    data_files="hf://datasets/sherry0213/MedCRAFT/medcraft_sft_train.jsonl",
    split="train"
)

print("Loading SFT eval split...")
sft_eval_raw = load_dataset(
    "json",
    data_files="hf://datasets/sherry0213/MedCRAFT/medcraft_sft_eval.jsonl",
    split="train"
)

print(f"\nSFT train size: {len(sft_train_raw):,}")
print(f"SFT eval size:  {len(sft_eval_raw):,}")
print(f"Columns: {sft_train_raw.column_names}")

Loading SFT train split...


medcraft_sft_train.jsonl:   0%|          | 0.00/28.6M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading SFT eval split...


medcraft_sft_eval.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]


SFT train size: 38,189
SFT eval size:  393
Columns: ['diff', 'instruction', 'response']


In [4]:
# Inspect a sample row and check column names
sample = dict(sft_train_raw[0])
print("=== Sample SFT Row ===")
for k, v in sample.items():
    print(f"[{k}]: {str(v)[:300]}")
    print()

# Check difficulty distribution if 'diff' column exists
df_sft_raw = sft_train_raw.to_pandas()
if "diff" in df_sft_raw.columns:
    print("\nDifficulty distribution:")
    print(df_sft_raw["diff"].value_counts().sort_index())

# Detect correct column names (handle both instruction/input naming variants)
cols = sft_train_raw.column_names
INSTRUCTION_COL = "instruction" if "instruction" in cols else "input"
RESPONSE_COL = "response" if "response" in cols else "output"
print(f"\nUsing instruction column: '{INSTRUCTION_COL}'")
print(f"Using response column: '{RESPONSE_COL}'")

print(f"\nAvg instruction length (chars): {df_sft_raw[INSTRUCTION_COL].str.len().mean():.0f}")
print(f"Avg response length (chars):    {df_sft_raw[RESPONSE_COL].str.len().mean():.0f}")

=== Sample SFT Row ===
[diff]: 2

[instruction]: List specific foods, such as prunes and pears, that can soften stools for a 1.2-year-old, using concise and easy-to-understand language.

[response]: Prunes and prune juice help soften stools.


Difficulty distribution:
diff
0     4386
1     4357
10     683
11     467
12     244
2     4257
3     4863
4     4821
5     4624
6     6243
7     1344
8     1034
9      866
Name: count, dtype: int64

Using instruction column: 'instruction'
Using response column: 'response'

Avg instruction length (chars): 288
Avg response length (chars):    404


In [5]:
# Clean and subset SFT dataset

# Using 10k of 38k samples
SUBSET_SIZE = 10000

df_sft = sft_train_raw.to_pandas()

# Drop nulls and empty strings
df_sft = df_sft.dropna(subset=[INSTRUCTION_COL, RESPONSE_COL])
df_sft = df_sft[df_sft[INSTRUCTION_COL].str.strip() != ""]
df_sft = df_sft[df_sft[RESPONSE_COL].str.strip() != ""]

# Drop very short responses (likely malformed)
df_sft = df_sft[df_sft[RESPONSE_COL].str.len() >= 30]

print(f"After cleaning: {len(df_sft):,} rows")

# Stratified sample by difficulty if available
if "diff" in df_sft.columns:
    df_sft_subset = (
        df_sft.groupby("diff", group_keys=False)
        .apply(lambda x: x.sample(frac=min(SUBSET_SIZE / len(df_sft), 1.0), random_state=42))
        .reset_index(drop=True)
    )
    if len(df_sft_subset) > SUBSET_SIZE:
        df_sft_subset = df_sft_subset.sample(n=SUBSET_SIZE, random_state=42).reset_index(drop=True)
else:
    df_sft_subset = df_sft.sample(n=min(SUBSET_SIZE, len(df_sft)), random_state=42).reset_index(drop=True)

print(f"SFT subset size: {len(df_sft_subset):,}")

After cleaning: 38,108 rows
SFT subset size: 10,000


/tmp/ipykernel_58/2677619208.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=min(SUBSET_SIZE / len(df_sft), 1.0), random_state=42))


In [6]:
# Format SFT dataset using Alpaca-style prompt
SFT_PREAMBLE = (
    "You are an experienced and knowledgeable medical professional. "
    "Provide clear, factual, and helpful medical information."
)

def format_sft_row(row):
    text = (
        f"{SFT_PREAMBLE}\n\n"
        f"### Instruction:\n{row[INSTRUCTION_COL]}\n\n"
        f"### Response:\n{row[RESPONSE_COL]}"
    )
    out = {"text": text, "instruction": row[INSTRUCTION_COL], "response": row[RESPONSE_COL]}
    if "diff" in row:
        out["diff"] = row["diff"]
    return out

sft_formatted = df_sft_subset.apply(format_sft_row, axis=1, result_type="expand")

print("Formatted sample (first 500 chars):")
print(sft_formatted.iloc[0]["text"][:500])
print("...")

# format eval set
df_eval = sft_eval_raw.to_pandas()
df_eval = df_eval.dropna(subset=[INSTRUCTION_COL, RESPONSE_COL])
df_eval = df_eval[df_eval[RESPONSE_COL].str.len() >= 30]
sft_eval_formatted = df_eval.apply(format_sft_row, axis=1, result_type="expand")

Formatted sample (first 500 chars):
You are an experienced and knowledgeable medical professional. Provide clear, factual, and helpful medical information.

### Instruction:
What are the biological mechanisms that differentiate the pathogens causing tetanus and COVID-19?

### Response:
Tetanus is caused by *Clostridium tetani*, a spore-forming anaerobic bacterium that produces the neurotoxin tetanospasmin, which blocks inhibitory neurotransmitters in the central nervous system, leading to spastic paralysis. COVID-19, caused by *SA
...


In [7]:
# Save SFT dataset
sft_train_hf = Dataset.from_pandas(sft_formatted.reset_index(drop=True))
sft_eval_hf = Dataset.from_pandas(sft_eval_formatted.reset_index(drop=True))

sft_dict = DatasetDict({"train": sft_train_hf, "validation": sft_eval_hf})
sft_dict.save_to_disk(os.path.join(OUTPUT_DIR, "sft_dataset"))

# Save CSV preview 
sft_formatted.to_csv(os.path.join(OUTPUT_DIR, "sft_dataset_preview.csv"), index=False)

print(f"SFT dataset saved to {OUTPUT_DIR}/sft_dataset")
print(f"  Train: {len(sft_train_hf):,} samples")
print(f"  Validation: {len(sft_eval_hf):,} samples")
print(f"CSV preview saved to {OUTPUT_DIR}/sft_dataset_preview.csv")

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/393 [00:00<?, ? examples/s]

SFT dataset saved to /kaggle/working/sft_dataset
  Train: 10,000 samples
  Validation: 393 samples
CSV preview saved to /kaggle/working/sft_dataset_preview.csv


## 2. DPO Dataset - sherry0213/MedCRAFT (DPO split)

In [8]:
# Load DPO dataset
print("Loading DPO dataset: sherry0213/MedCRAFT (medcraft_dpo_2k.jsonl)...")
dpo_raw = load_dataset(
    "json",
    data_files="hf://datasets/sherry0213/MedCRAFT/medcraft_dpo_2k.jsonl",
    split="train"
)

print(f"DPO dataset size: {len(dpo_raw):,}")
print(f"Columns: {dpo_raw.column_names}")
print("\n=== Sample DPO Row ===")
for k, v in dict(dpo_raw[0]).items():
    print(f"[{k}]: {str(v)[:250]}")
    print()

Loading DPO dataset: sherry0213/MedCRAFT (medcraft_dpo_2k.jsonl)...


medcraft_dpo_2k.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DPO dataset size: 2,172
Columns: ['prompt', 'chosen', 'rejected']

=== Sample DPO Row ===
[prompt]: How does the frequency of ejaculation affect the composition and appearance of semen?

[chosen]: Frequent ejaculation can reduce semen volume and sperm concentration, making it appear thinner and more transparent.

[rejected]: Daily ejaculation could lead to a temporary decline in sperm count but does not significantly alter long-term fertility or semen quality.



In [9]:
# Clean and format DPO dataset
df_dpo = dpo_raw.to_pandas()

# Detect prompt column (could be 'instruction' or 'prompt')
dpo_cols = df_dpo.columns.tolist()
PROMPT_COL = "instruction" if "instruction" in dpo_cols else "prompt"
print(f"Prompt column detected: '{PROMPT_COL}'")
print(f"Columns: {dpo_cols}")

# Verify chosen and rejected columns exist
assert "chosen" in dpo_cols, "ERROR: 'chosen' column not found in DPO dataset"
assert "rejected" in dpo_cols, "ERROR: 'rejected' column not found in DPO dataset"

# Clean
df_dpo = df_dpo.dropna(subset=[PROMPT_COL, "chosen", "rejected"])
df_dpo = df_dpo[df_dpo["chosen"].str.strip() != ""]
df_dpo = df_dpo[df_dpo["rejected"].str.strip() != ""]
df_dpo = df_dpo[df_dpo["chosen"] != df_dpo["rejected"]]  # remove zero-signal rows

print(f"\nRows after cleaning: {len(df_dpo):,}")
print(f"Avg chosen length (chars): {df_dpo['chosen'].str.len().mean():.0f}")
print(f"Avg rejected length (chars): {df_dpo['rejected'].str.len().mean():.0f}")

# Format for TRL DPOTrainer - Alpaca-style prompt for base model
def format_dpo_row(row):
    prompt = (
        f"{SFT_PREAMBLE}\n\n"
        f"### Instruction:\n{row[PROMPT_COL]}\n\n"
        f"### Response:\n"
    )
    return {
        "prompt": prompt,
        "chosen": row["chosen"],
        "rejected": row["rejected"],
    }

dpo_formatted = df_dpo.apply(format_dpo_row, axis=1, result_type="expand")

print("\nFormatted DPO sample:")
row0 = dpo_formatted.iloc[0]
print(f"PROMPT:\n{row0['prompt'][:300]}")
print(f"\nCHOSEN: {row0['chosen'][:200]}")
print(f"\nREJECTED: {row0['rejected'][:200]}")

Prompt column detected: 'prompt'
Columns: ['prompt', 'chosen', 'rejected']

Rows after cleaning: 2,168
Avg chosen length (chars): 420
Avg rejected length (chars): 334

Formatted DPO sample:
PROMPT:
You are an experienced and knowledgeable medical professional. Provide clear, factual, and helpful medical information.

### Instruction:
How does the frequency of ejaculation affect the composition and appearance of semen?

### Response:


CHOSEN: Frequent ejaculation can reduce semen volume and sperm concentration, making it appear thinner and more transparent.

REJECTED: Daily ejaculation could lead to a temporary decline in sperm count but does not significantly alter long-term fertility or semen quality.


In [10]:
# Save DPO dataset
dpo_hf = Dataset.from_pandas(dpo_formatted.reset_index(drop=True))
dpo_split = dpo_hf.train_test_split(test_size=0.1, seed=42)
dpo_split.save_to_disk(os.path.join(OUTPUT_DIR, "dpo_dataset"))

dpo_formatted.to_csv(os.path.join(OUTPUT_DIR, "dpo_dataset_preview.csv"), index=False)

print(f"DPO dataset saved to {OUTPUT_DIR}/dpo_dataset")
print(f"  Train: {len(dpo_split['train']):,} samples")
print(f"  Validation: {len(dpo_split['test']):,} samples")
print(f"CSV preview saved to {OUTPUT_DIR}/dpo_dataset_preview.csv")

Saving the dataset (0/1 shards):   0%|          | 0/1951 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/217 [00:00<?, ? examples/s]

DPO dataset saved to /kaggle/working/dpo_dataset
  Train: 1,951 samples
  Validation: 217 samples
CSV preview saved to /kaggle/working/dpo_dataset_preview.csv


## 3. Test Prompts

In [11]:
# Define 10 test prompts

TEST_PROMPTS = [
    {
        "id": 1,
        "category": "Chronic Disease",
        "question": "What are the early warning signs of Type 2 diabetes and how is it diagnosed?",
        "gold_answer": ""
    },
    {
        "id": 2,
        "category": "Pharmacology",
        "question": "What is the mechanism of action of metformin and why is it the first-line treatment for Type 2 diabetes?",
        "gold_answer": ""
    },
    {
        "id": 3,
        "category": "Emergency Medicine",
        "question": "What are the differences between ischemic and hemorrhagic stroke, and what are the emergency treatment options for each?",
        "gold_answer": ""
    },
    {
        "id": 4,
        "category": "Pathophysiology",
        "question": "How does chronic hypertension damage the kidneys over time, and what lifestyle changes can slow this progression?",
        "gold_answer": ""
    },
    {
        "id": 5,
        "category": "Pharmacology",
        "question": "What are the common side effects of statins, and which patient populations require the closest monitoring?",
        "gold_answer": ""
    },
    {
        "id": 6,
        "category": "Cardiology",
        "question": "Explain the difference between systolic and diastolic heart failure, including their causes and typical clinical presentations.",
        "gold_answer": ""
    },
    {
        "id": 7,
        "category": "Rheumatology",
        "question": "What is rheumatoid arthritis, how does it differ pathologically from osteoarthritis, and what are the main treatment approaches?",
        "gold_answer": ""
    },
    {
        "id": 8,
        "category": "Critical Care",
        "question": "What are the signs and symptoms of sepsis, and describe the Sepsis Hour-1 Bundle protocol used in emergency management?",
        "gold_answer": ""
    },
    {
        "id": 9,
        "category": "Preventive Medicine",
        "question": "What is the recommended screening schedule for colorectal cancer, and which patients qualify for earlier screening?",
        "gold_answer": ""
    },
    {
        "id": 10,
        "category": "Psychiatry",
        "question": "How do SSRIs work in treating depression, and what are the key clinical differences between sertraline, fluoxetine, and escitalopram?",
        "gold_answer": ""
    },
]

print(f"Defined {len(TEST_PROMPTS)} test prompts:")
for p in TEST_PROMPTS:
    print(f"  [{p['id']}] ({p['category']}) {p['question'][:80]}")

Defined 10 test prompts:
  [1] (Chronic Disease) What are the early warning signs of Type 2 diabetes and how is it diagnosed?
  [2] (Pharmacology) What is the mechanism of action of metformin and why is it the first-line treatm
  [3] (Emergency Medicine) What are the differences between ischemic and hemorrhagic stroke, and what are t
  [4] (Pathophysiology) How does chronic hypertension damage the kidneys over time, and what lifestyle c
  [5] (Pharmacology) What are the common side effects of statins, and which patient populations requi
  [6] (Cardiology) Explain the difference between systolic and diastolic heart failure, including t
  [7] (Rheumatology) What is rheumatoid arthritis, how does it differ pathologically from osteoarthri
  [8] (Critical Care) What are the signs and symptoms of sepsis, and describe the Sepsis Hour-1 Bundle
  [9] (Preventive Medicine) What is the recommended screening schedule for colorectal cancer, and which pati
  [10] (Psychiatry) How do SSRIs work in tr

#### Gold answers from GPT-5.5.

System Prompt: 
`You are an experienced and knowledgeable medical professional. Answer the following question accurately and concisely. Cover the key clinical facts and mechanisms in 2–5 sentences. Match the complexity of the question - simple questions get short, direct answers, and complex multi-part questions may use a brief structured explanation or bullet points where appropriate. Do not pad or repeat yourself.`

In [12]:
GOLD_ANSWERS = {
    1: """
Early warning signs of type 2 diabetes can be subtle and include increased thirst, frequent urination, fatigue, blurred vision, slow-healing cuts/infections, tingling or numbness in the feet/hands, and sometimes unexplained weight loss. Mechanistically, insulin resistance keeps glucose in the blood instead of moving it efficiently into cells, causing high blood sugar and osmotic fluid loss through urine.

Diagnosis is made with blood tests: HbA1c ≥ 6.5%, fasting plasma glucose ≥ 126 mg/dL, 2-hour OGTT glucose ≥ 200 mg/dL, or random glucose ≥ 200 mg/dL with classic symptoms; in people without clear symptoms, abnormal results are usually confirmed with repeat testing.
""",
    2: """
Metformin mainly reduces hepatic gluconeogenesis and improves peripheral insulin sensitivity, partly through effects on cellular energy metabolism/AMPK pathways; it lowers glucose without directly stimulating insulin secretion, so hypoglycemia risk is low. It is historically first-line for type 2 diabetes because it is effective, inexpensive, weight-neutral or modestly weight-reducing, generally safe, and has long clinical experience. Current guidelines increasingly individualize first therapy—especially adding or prioritizing GLP-1 receptor agonists or SGLT2 inhibitors in patients with cardiovascular, kidney, heart failure, or obesity-related indications—but metformin remains a common foundation when tolerated and not contraindicated.
""",
    3: """
An ischemic stroke is caused by arterial blockage, cutting off blood flow and oxygen to brain tissue; emergency treatment focuses on reperfusion with IV thrombolysis such as alteplase/tenecteplase when eligible, and mechanical thrombectomy for selected large-vessel occlusions. A hemorrhagic stroke is caused by bleeding into or around the brain, causing mass effect, raised intracranial pressure, and toxic blood injury; treatment focuses on stopping bleeding and limiting expansion with rapid blood-pressure control, reversal of anticoagulants/antiplatelet effects when appropriate, neurosurgical/endovascular intervention for selected bleeds or aneurysms, and intracranial-pressure management. Because treatments differ sharply, suspected stroke requires urgent brain imaging—usually non-contrast CT—to distinguish ischemia from hemorrhage before giving clot-busting therapy.
""",
    4: """
Chronic hypertension damages the kidneys by keeping pressure high inside small renal arteries and glomeruli, causing vessel wall thickening, scarring, reduced filtration, and protein leakage; over time this can progress to chronic kidney disease. The key way to slow progression is tight blood pressure control, usually with prescribed medications plus lifestyle changes. Helpful changes include lower sodium intake/DASH-style diet, weight control, regular physical activity, quitting smoking, limiting alcohol, managing stress, and controlling diabetes if present. Sodium reduction is especially important because high salt intake raises blood pressure and further stresses kidney vessels.
""",
    5: """
Common statin side effects include muscle aches/weakness, mild gastrointestinal symptoms, headache, and transient liver enzyme elevation; rare but serious effects are myopathy/rhabdomyolysis, clinically significant liver injury, and a small increase in blood glucose/diabetes risk. Patients needing closest monitoring are those on high-dose statins, older/frail patients, people with kidney or liver disease, hypothyroidism, prior muscle disease/statin intolerance, heavy alcohol use, or interacting drugs such as fibrates, cyclosporine, macrolides, azole antifungals, protease inhibitors, and some CYP3A4/OATP inhibitors. CK is generally checked when muscle symptoms occur, while liver enzymes are checked at baseline and if symptoms suggest hepatotoxicity
""",
    6: """
Systolic heart failure—now called HFrEF—means the ventricle is weak and cannot contract effectively, so ejection fraction is reduced, typically ≤40%; common causes include prior myocardial infarction, ischemic heart disease, dilated cardiomyopathy, myocarditis, and toxic injury such as alcohol or chemotherapy. Diastolic heart failure—or HFpEF—means the ventricle is stiff and cannot relax/fill properly, so EF is usually ≥50% despite high filling pressures; it is commonly linked to long-standing hypertension, aging, obesity, diabetes, atrial fibrillation, and left ventricular hypertrophy. Both present with congestion symptoms such as exertional dyspnea, orthopnea, fatigue, pulmonary crackles, edema, and exercise intolerance, but HFrEF often reflects “pump failure,” while HFpEF often presents with symptoms triggered by exertion or hypertension because filling pressures rise quickly.
""",
    7: """
Rheumatoid arthritis is a systemic autoimmune inflammatory arthritis in which immune-mediated synovitis forms pannus that erodes cartilage and bone, usually causing symmetric small-joint pain, swelling, and prolonged morning stiffness. Osteoarthritis is mainly a degenerative/mechanical cartilage disease with cartilage loss, subchondral bone change, and osteophytes, typically worse with use and less systemically inflammatory. RA treatment aims to suppress inflammation and prevent joint destruction: early DMARDs, especially methotrexate when appropriate, escalation to biologic or targeted synthetic DMARDs if needed, short-term NSAIDs or glucocorticoids for symptom control, plus exercise, smoking cessation, vaccines, and monitoring for drug toxicity.
""",
    8: """
Sepsis is life-threatening organ dysfunction from a dysregulated response to infection; warning signs include fever or hypothermia, tachycardia, tachypnea, confusion, hypotension, reduced urine output, mottled/cool skin, and signs of a source infection. The Surviving Sepsis Campaign Hour-1 Bundle is started immediately: measure lactate and recheck if elevated, obtain blood cultures before antibiotics if this does not delay care, give broad-spectrum IV antibiotics, give 30 mL/kg crystalloid for hypotension or lactate ≥4 mmol/L, and start vasopressors if hypotension persists to maintain MAP ≥65 mmHg. Early management works by rapidly treating infection, restoring perfusion, and preventing shock-related organ failure.
""",
    9: """
For average-risk adults, colorectal cancer screening is recommended from age 45 to 75; from 76 to 85, screening is individualized based on health, life expectancy, prior screening, and preferences, and it is generally stopped after 85. Options include annual FIT or high-sensitivity stool blood testing, stool DNA-FIT every 1–3 years, CT colonography every 5 years, flexible sigmoidoscopy every 5 years, or colonoscopy every 10 years; any abnormal non-colonoscopy test needs diagnostic colonoscopy. Earlier or more frequent screening is needed for patients with personal history of colorectal cancer or adenomatous polyps, inflammatory bowel disease, family history of colorectal cancer/polyps, hereditary syndromes such as Lynch syndrome or FAP, or prior abdominal/pelvic radiation.
""",
    10: """
SSRIs treat depression by blocking the serotonin transporter (SERT), increasing synaptic serotonin; the clinical antidepressant effect usually takes weeks because downstream receptor, circuit, and neuroplastic changes matter more than the immediate serotonin rise. Sertraline is a balanced, commonly used SSRI with a ~26-hour half-life and is often favored when cardiac comorbidity or broad anxiety coverage matters; GI upset/diarrhea is relatively common. Fluoxetine is more activating, has a very long half-life with active norfluoxetine, so it causes fewer discontinuation symptoms but more prolonged drug interactions and is a stronger CYP2D6 inhibitor. Escitalopram is often well tolerated with relatively few CYP interactions, but dose caution is needed in older adults/hepatic impairment and in patients at risk for QT prolongation.
""",
}

# Inject gold answers into TEST_PROMPTS
for p in TEST_PROMPTS:
    p["gold_answer"] = GOLD_ANSWERS[p["id"]].strip()

# Verify all filled
missing = [p["id"] for p in TEST_PROMPTS if len(p["gold_answer"]) < 20]
if missing:
    print(f"WARNING: Gold answers appear empty or too short for IDs: {missing}")
    print("Fill in GOLD_ANSWERS dict above before running the rest of this notebook.")
else:
    print("All 10 gold answers filled in successfully.")
    for p in TEST_PROMPTS:
        print(f"  Q{p['id']}: {len(p['gold_answer'])} chars")

# Save test_prompts.json
with open(os.path.join(OUTPUT_DIR, "test_prompts.json"), "w") as f:
    json.dump(TEST_PROMPTS, f, indent=2)
print(f"\nSaved to {OUTPUT_DIR}/test_prompts.json")

All 10 gold answers filled in successfully.
  Q1: 674 chars
  Q2: 745 chars
  Q3: 878 chars
  Q4: 690 chars
  Q5: 757 chars
  Q6: 892 chars
  Q7: 757 chars
  Q8: 724 chars
  Q9: 783 chars
  Q10: 839 chars

Saved to /kaggle/working/test_prompts.json


## 4. Load Base Model and Run Baseline Inference

In [14]:
# Load TinyLlama-1.1B base model
# Prompts formatted in Alpaca style (### Instruction / ### Response)

MODEL_ID = "TinyLlama/TinyLlama_v1.1"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Loading model in float16 on {DEVICE}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

param_count = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded. Parameters: {param_count:.2f}B")
if torch.cuda.is_available():
    used_vram = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM used: {used_vram:.2f} GB")

Loading tokenizer: TinyLlama/TinyLlama_v1.1


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Model loaded. Parameters: 1.10B
VRAM used: 1.01 GB


In [15]:
# Define inference function
# Greedy decoding (do_sample=False) for reproducibility across all trials

def generate_response(question, max_new_tokens=512):
    formatted = (
        f"{SFT_PREAMBLE}\n\n"
        f"### Instruction:\n{question}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens (exclude the prompt tokens)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Quick sanity check
test_response = generate_response("What is paracetamol used for?", max_new_tokens=100)
print(f"Sanity check response: {test_response[:200]}")

Error during conversion: AttributeError("'str' object has no attribute 'decode'")
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The

Sanity check response: Paracetamol is used to relieve pain and fever.

### Instruction:
What is ibuprofen used for?

### Response:

Ibuprofen is used to relieve pain and fever.

### Instruction:
What is aspirin used for?

#


In [16]:
# Run base model on all 10 test prompts

missing = [p["id"] for p in TEST_PROMPTS if len(p["gold_answer"]) < 20]
if missing:
    raise RuntimeError(
        f"Gold answers not filled for prompt IDs: {missing}. "
        "Complete Cell 12 first."
    )

print("Running base model on 10 test prompts (greedy decoding, max_new_tokens=512)...\n")

results = []
for p in tqdm(TEST_PROMPTS, desc="Baseline inference"):
    response = generate_response(p["question"])
    results.append({
        "id": p["id"],
        "category": p["category"],
        "question": p["question"],
        "gold_answer": p["gold_answer"],
        "base_model_response": response,
    })
    print(f"Q{p['id']} ({p['category']}): {response[:120]}...\n")

df_results = pd.DataFrame(results)
print(f"Inference complete. {len(df_results)} prompts processed.")

Running base model on 10 test prompts (greedy decoding, max_new_tokens=512)...



Baseline inference:  10%|█         | 1/10 [00:18<02:50, 18.96s/it]

Q1 (Chronic Disease): The early warning signs of Type 2 diabetes are:

  * **High blood sugar levels:** Blood sugar levels are high.
  * **Wei...



Baseline inference:  20%|██        | 2/10 [00:37<02:31, 18.96s/it]

Q2 (Pharmacology): Metformin is a biguanide that is used to treat Type 2 diabetes. It is the first-line treatment for Type 2 diabetes. Metf...



...

Q8 (Critical Care): Sepsis is a life-threatening condition that can result from an infection. Sepsis is a medical emergency that requires im...



Baseline inference:  90%|█████████ | 9/10 [02:51<00:18, 18.99s/it]

Q9 (Preventive Medicine): The recommended screening schedule for colorectal cancer is every 10 years for men and every 5 years for women. Patients...



Baseline inference: 100%|██████████| 10/10 [03:09<00:00, 19.00s/it]

Q10 (Psychiatry): SSRIs work by increasing the levels of serotonin in the brain. Serotonin is a neurotransmitter that is involved in the r...

Inference complete. 10 prompts processed.



## 5. Compute BLEU + BERTScore

In [17]:
# Compute BLEU scores
# Reporting both sentence-level (per prompt) and corpus-level

bleu_metric = BLEU(effective_order=True)

sentence_bleu_scores = []
for _, row in df_results.iterrows():
    result = bleu_metric.sentence_score(
        hypothesis=row["base_model_response"],
        references=[row["gold_answer"]],
    )
    sentence_bleu_scores.append(result.score)

df_results["bleu_score"] = sentence_bleu_scores

# Corpus-level BLEU is the standard reported metric
corpus_bleu = bleu_metric.corpus_score(
    hypotheses=df_results["base_model_response"].tolist(),
    references=[df_results["gold_answer"].tolist()],
)

print("=== BLEU Scores ===")
print(f"Corpus BLEU:          {corpus_bleu.score:.4f}")
print(f"Mean sentence BLEU:   {np.mean(sentence_bleu_scores):.4f}")
print()
for i, score in enumerate(sentence_bleu_scores):
    print(f"  Q{i+1} ({df_results.iloc[i]['category']}): {score:.4f}")

=== BLEU Scores ===
Corpus BLEU:          1.1166
Mean sentence BLEU:   1.1342

  Q1 (Chronic Disease): 0.6397
  Q2 (Pharmacology): 0.7025
  Q3 (Emergency Medicine): 1.5801
  Q4 (Pathophysiology): 1.6697
  Q5 (Pharmacology): 0.7257
  Q6 (Cardiology): 0.3407
  Q7 (Rheumatology): 1.0058
  Q8 (Critical Care): 0.3501
  Q9 (Preventive Medicine): 3.2540
  Q10 (Psychiatry): 1.0738


In [18]:
# Compute BERTScore using roberta-large as backbone

print("Computing BERTScore with roberta-large backbone...")
print("This takes ~2-3 minutes on GPU.")

P, R, F1 = bert_score_fn(
    cands=df_results["base_model_response"].tolist(),
    refs=df_results["gold_answer"].tolist(),
    lang="en",
    model_type="roberta-large",
    verbose=True,
    device=DEVICE,
)

df_results["bertscore_precision"] = P.numpy()
df_results["bertscore_recall"] = R.numpy()
df_results["bertscore_f1"] = F1.numpy()

print("\n=== BERTScore (roberta-large) ===")
print(f"Mean Precision: {P.mean().item():.4f}")
print(f"Mean Recall:    {R.mean().item():.4f}")
print(f"Mean F1:        {F1.mean().item():.4f}")
print()
for i in range(len(df_results)):
    row = df_results.iloc[i]
    print(f"  Q{row['id']} ({row['category']}): P={P[i]:.4f} R={R[i]:.4f} F1={F1[i]:.4f}")

Computing BERTScore with roberta-large backbone...
This takes ~2-3 minutes on GPU.


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.59 seconds, 6.29 sentences/sec

=== BERTScore (roberta-large) ===
Mean Precision: 0.7649
Mean Recall:    0.7942
Mean F1:        0.7788

  Q1 (Chronic Disease): P=0.7410 R=0.7936 F1=0.7664
  Q2 (Pharmacology): P=0.7733 R=0.7837 F1=0.7784
  Q3 (Emergency Medicine): P=0.7817 R=0.7722 F1=0.7769
  Q4 (Pathophysiology): P=0.7349 R=0.8505 F1=0.7885
  Q5 (Pharmacology): P=0.7828 R=0.7798 F1=0.7813
  Q6 (Cardiology): P=0.7787 R=0.7799 F1=0.7793
  Q7 (Rheumatology): P=0.7613 R=0.7950 F1=0.7778
  Q8 (Critical Care): P=0.7129 R=0.7790 F1=0.7445
  Q9 (Preventive Medicine): P=0.8081 R=0.8156 F1=0.8118
  Q10 (Psychiatry): P=0.7745 R=0.7929 F1=0.7836


In [19]:
# summary table

print("=" * 80)
print("BASELINE RESULTS — BASE MODEL vs CHATGPT GOLD ANSWERS")
print("=" * 80)
print(f"Model: {MODEL_ID}")
print(f"Inference settings: greedy decoding, max_new_tokens=512, float16")
print()
print(f"Corpus BLEU:          {corpus_bleu.score:.4f}")
print(f"Mean sentence BLEU:   {np.mean(sentence_bleu_scores):.4f}")
print(f"Mean BERTScore P:     {P.mean().item():.4f}")
print(f"Mean BERTScore R:     {R.mean().item():.4f}")
print(f"Mean BERTScore F1:    {F1.mean().item():.4f}")
print()

# Per-prompt table
summary_df = df_results[["id", "category", "bleu_score", "bertscore_f1"]].copy()
summary_df["bleu_score"] = summary_df["bleu_score"].round(4)
summary_df["bertscore_f1"] = summary_df["bertscore_f1"].round(4)
print(summary_df.to_string(index=False))

BASELINE RESULTS — BASE MODEL vs CHATGPT GOLD ANSWERS
Model: TinyLlama/TinyLlama_v1.1
Inference settings: greedy decoding, max_new_tokens=512, float16

Corpus BLEU:          1.1166
Mean sentence BLEU:   1.1342
Mean BERTScore P:     0.7649
Mean BERTScore R:     0.7942
Mean BERTScore F1:    0.7788

 id            category  bleu_score  bertscore_f1
  1     Chronic Disease      0.6397        0.7664
  2        Pharmacology      0.7025        0.7784
  3  Emergency Medicine      1.5801        0.7769
  4     Pathophysiology      1.6697        0.7885
  5        Pharmacology      0.7257        0.7813
  6          Cardiology      0.3407        0.7793
  7        Rheumatology      1.0058        0.7778
  8       Critical Care      0.3501        0.7445
  9 Preventive Medicine      3.2540        0.8118
 10          Psychiatry      1.0738        0.7836


In [20]:
# sample responses

print("=" * 80)
print("SAMPLE RESPONSES FOR REPORT")
print("=" * 80)

for _, row in df_results.iterrows():
    print(f"\n[Q{row['id']}] {row['question']}")
    print("-" * 70)
    print(f"GOLD ANSWER (ChatGPT):\n{row['gold_answer'][:400]}")
    print(f"\nBASE MODEL RESPONSE:\n{row['base_model_response'][:400]}")
    print(f"\nBLEU: {row['bleu_score']:.4f}  |  BERTScore F1: {row['bertscore_f1']:.4f}")
    print()

SAMPLE RESPONSES FOR REPORT

[Q1] What are the early warning signs of Type 2 diabetes and how is it diagnosed?
----------------------------------------------------------------------
GOLD ANSWER (ChatGPT):
Early warning signs of type 2 diabetes can be subtle and include increased thirst, frequent urination, fatigue, blurred vision, slow-healing cuts/infections, tingling or numbness in the feet/hands, and sometimes unexplained weight loss. Mechanistically, insulin resistance keeps glucose in the blood instead of moving it efficiently into cells, causing high blood sugar and osmotic fluid loss through

BASE MODEL RESPONSE:
The early warning signs of Type 2 diabetes are:

  * **High blood sugar levels:** Blood sugar levels are high.
  * **Weight gain:** Weight gain is common.
  * **Increased thirst:** Thirst is common.
  * **Increased hunger:** Hunger is common.
  * **Increased urination:** Urination is common.
  * **Increased fatigue:** Fatigue is common.
  * **Increased tiredness:** Tire

In [21]:
# Save all outputs

# Full results CSV
df_results.to_csv(os.path.join(OUTPUT_DIR, "baseline_results.csv"), index=False)

# Compact JSON summary for report tables
summary = {
    "model": MODEL_ID,
    "inference_settings": "greedy, max_new_tokens=512, float16",
    "corpus_bleu": round(corpus_bleu.score, 4),
    "mean_sentence_bleu": round(float(np.mean(sentence_bleu_scores)), 4),
    "mean_bertscore_precision": round(P.mean().item(), 4),
    "mean_bertscore_recall": round(R.mean().item(), 4),
    "mean_bertscore_f1": round(F1.mean().item(), 4),
    "per_prompt": [
        {
            "id": int(row["id"]),
            "category": row["category"],
            "bleu": round(float(row["bleu_score"]), 4),
            "bertscore_f1": round(float(row["bertscore_f1"]), 4),
        }
        for _, row in df_results.iterrows()
    ]
}

with open(os.path.join(OUTPUT_DIR, "baseline_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "test_prompts.json"), "w") as f:
    json.dump(TEST_PROMPTS, f, indent=2)

print("Files saved to /kaggle/working/:")
print("  baseline_results.csv   -> full table (share with Person 2 and 3)")
print("  baseline_summary.json  -> compact summary for report tables")
print("  test_prompts.json      -> 10 prompts with gold answers (share with all)")
print("  sft_dataset/           -> preprocessed SFT dataset (share with Person 2)")
print("  dpo_dataset/           -> preprocessed DPO dataset (share with Person 3)")

# List output files
import glob
all_files = glob.glob(os.path.join(OUTPUT_DIR, "*"))
print("\nAll output files:")
for f in sorted(all_files):
    size = os.path.getsize(f) if os.path.isfile(f) else sum(
        os.path.getsize(os.path.join(root, file))
        for root, _, files in os.walk(f) for file in files
    )
    print(f"  {os.path.basename(f)}: {size/1e6:.2f} MB")

Files saved to /kaggle/working/:
  baseline_results.csv   -> full table (share with Person 2 and 3)
  baseline_summary.json  -> compact summary for report tables
  test_prompts.json      -> 10 prompts with gold answers (share with all)
  sft_dataset/           -> preprocessed SFT dataset (share with Person 2)
  dpo_dataset/           -> preprocessed DPO dataset (share with Person 3)

All output files:
  baseline_results.csv: 0.03 MB
  baseline_summary.json: 0.00 MB
  dpo_dataset: 2.44 MB
  dpo_dataset_preview.csv: 2.43 MB
  sft_dataset: 16.28 MB
  sft_dataset_preview.csv: 15.61 MB
  test_prompts.json: 0.01 MB


In [23]:
# Dataset statistics

print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

print("\nSFT DATASET: sherry0213/MedCRAFT")
try:
    from datasets import load_from_disk
    sft_loaded = load_from_disk(os.path.join(OUTPUT_DIR, "sft_dataset"))
    print(f"  Train samples:      {len(sft_loaded['train']):,}")
    print(f"  Validation samples: {len(sft_loaded['validation']):,}")
    print(f"  Columns:            {sft_loaded['train'].column_names}")
except Exception as e:
    print(f"  (load error: {e})")

print("\nDPO DATASET: sherry0213/MedCRAFT (DPO split)")
try:
    dpo_loaded = load_from_disk(os.path.join(OUTPUT_DIR, "dpo_dataset"))
    print(f"  Train samples:      {len(dpo_loaded['train']):,}")
    print(f"  Validation samples: {len(dpo_loaded['test']):,}")
    print(f"  Columns:            {dpo_loaded['train'].column_names}")
except Exception as e:
    print(f"  (load error: {e})")

print("\nBASELINE RESULTS SUMMARY")
with open(os.path.join(OUTPUT_DIR, "baseline_summary.json")) as f:
    bs = json.load(f)
print(f"  Corpus BLEU:          {bs['corpus_bleu']}")
print(f"  Mean sentence BLEU:   {bs['mean_sentence_bleu']}")
print(f"  Mean BERTScore F1:    {bs['mean_bertscore_f1']}")

DATASET STATISTICS

SFT DATASET: sherry0213/MedCRAFT
  Train samples:      10,000
  Validation samples: 393
  Columns:            ['text', 'instruction', 'response', 'diff']

DPO DATASET: sherry0213/MedCRAFT (DPO split)
  Train samples:      1,951
  Validation samples: 217
  Columns:            ['prompt', 'chosen', 'rejected']

BASELINE RESULTS SUMMARY
  Corpus BLEU:          1.1166
  Mean sentence BLEU:   1.1342
  Mean BERTScore F1:    0.7788
